## BRINGING IN COMPETITION'S TRAIN DATA
We need this in order to align our data in same format before merging.

In [100]:
import pandas as pd
import os
os.getcwd()

'C:\\Users\\user\\Downloads\\slm project\\check-points'

In [101]:
import zipfile

with zipfile.ZipFile("C:\\Users\\user\\Downloads\\slm project\\data\\agriculture-climate-slm-challenge.zip") as z:
    print(z.namelist())

['baseline_submission.csv', 'dataset-metadata.json', 'documents.csv', 'sample_submission.csv', 'test_questions.csv', 'train_qa.csv']


In [104]:
import pandas as pd
import zipfile

with zipfile.ZipFile("C:\\Users\\user\\Downloads\\slm project\\data\\agriculture-climate-slm-challenge.zip") as z:
    
    train_qa = pd.read_csv(z.open("train_qa.csv"))

In [105]:
train_qa.head()

,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5


In [106]:
train_qa.shape

(45, 7)

## BRINGING IN OUR GENERATED ANSWERS

In [107]:
# Turn the .jsonl file to dataframe and view

df = pd.read_json("C:\\Users\\user\\Downloads\\slm project\\fao_cgiar_checkpoint.jsonl", lines=True)

print(df.head())

                                            question               topic  \
0  What disease does Xanthomonas vasicola pv. mus...       crop_diseases   
1  How can we handle disputes with neighbors with...             general   
2  If I replace half of the fodder I collect with...  climate_adaptation   
3  How can the Viazi Soko digital platform help m...             general   
4  Which vegetables are mentioned as being grown ...             general   

        crop  agro_zone  document_id  \
0    general  sub_humid  doc_dis_007   
1    general  semi_arid  doc_gen_016   
2  livestock   highland  doc_cli_016   
3    general   highland  doc_gen_022   
4    general  semi_arid  doc_gen_026   

                                    reference_answer  q_index  
0                It causes bacterial wilt on banana.        0  
1  You can manage disagreements informally, using...        0  
2            Yes, replacing 50 % of collected fodder        0  
3                             Viazi Soko offer

In [108]:
df.head()

,question,topic,crop,agro_zone,document_id,reference_answer,q_index
0,What disease does Xanthomonas vasicola pv. mus...,crop_diseases,general,sub_humid,doc_dis_007,It causes bacterial wilt on banana.,0
1,How can we handle disputes with neighbors with...,general,general,semi_arid,doc_gen_016,"You can manage disagreements informally, using...",0
2,If I replace half of the fodder I collect with...,climate_adaptation,livestock,highland,doc_cli_016,"Yes, replacing 50 % of collected fodder",0
3,How can the Viazi Soko digital platform help m...,general,general,highland,doc_gen_022,Viazi Soko offers e‑ad,0
4,Which vegetables are mentioned as being grown ...,general,general,semi_arid,doc_gen_026,Kale and Swiss,0


In [109]:
df.shape

(953, 7)

In [110]:
df.isna().sum()

question            0
topic               0
crop                0
agro_zone           0
document_id         0
reference_answer    0
q_index             0
dtype: int64

In [112]:
df.iloc[497]['reference_answer']

'The varieties ICGV86124, Fleur\u202f11, and JL\u202f24 produce high‑quality seed.'

### Ooooops!!!!... unicode!!!1...

Got to correct that.

In [113]:
# Cleaning the dataset 

import json
import re
import unicodedata


TYPOGRAPHIC_REPLACEMENTS = {
    "\u2011": "-", "\u2013": "-", "\u2014": "-",
    "\u202f": " ", "\u00a0": " ",
    "\u2018": "'", "\u2019": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2026": "...",
}

def normalize_text(text: str) -> str:
    for bad, good in TYPOGRAPHIC_REPLACEMENTS.items():
        text = text.replace(bad, good)
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_invalid(text) -> bool:
    if not text:
        return True
    lowered = str(text).strip().lower()
    if lowered in {"skip", "n/a", "none", ""}:
        return True
    if "<question>" in lowered or "<answer>" in lowered:
        return True
    if len(str(text).strip()) < 8:
        return True
    return False

good_rows = []
dropped = 0

with open("C:\\Users\\user\\Downloads\\slm project\\fao_cgiar_checkpoint.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        question = row.get("question", "")
        answer = row.get("reference_answer", "")

        if is_invalid(question) or is_invalid(answer):
            dropped += 1
            continue

        row["question"] = normalize_text(question)
        row["reference_answer"] = normalize_text(answer)
        good_rows.append(row)

print(f"Kept {len(good_rows)} rows, dropped {dropped} invalid rows")

import pandas as pd
df = pd.DataFrame(good_rows)
df = df.drop(columns=["q_index"], errors="ignore")

Kept 783 rows, dropped 170 invalid rows


## MERGING BOTH DATASETS

In [118]:
train = pd.concat([train_qa, df])
train.shape

(828, 7)

In [119]:
train.head()

,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1.0
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2.0
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3.0
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4.0
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5.0


## CHECK FOR DUPLICATES

In [120]:
train.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
778    False
779    False
780    False
781    False
782    False
Length: 828, dtype: bool

In [121]:
duplicate_count = train.duplicated().sum()
print("Total duplicate rows:", duplicate_count)

Total duplicate rows: 0


In [24]:
train.iloc[541]['question']

'How many households in Malawi received remittances in 2016/17?'

## BRINGING TO ORDER THE 'QUESTIONID' COLUMN

In [126]:
train = train.reset_index(drop=True)
train["QuestionId"] = range(1, len(train) + 1)

print(train[["QuestionId", "question"]].head())
print(f"Range: {train['QuestionId'].min()} to {train['QuestionId'].max()}")

   QuestionId                                           question
0           1         Weevils in stored maize without chemicals?
1           2    Insurance paid but my field still failed — why?
2           3      Which cover crop helps between maize seasons?
3           4  Maize stalks lodging before harvest — nutrient...
4           5                       What are signs of bean rust?
Range: 1 to 828


In [128]:
train.tail()

,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
823,What was the purpose of the Innovation Follow-...,general,general,semi_arid,doc_gen_394,To gather firm-level data on innovation and re...,824
824,Does raising goats or poultry help improve the...,livestock,livestock,semi_arid,doc_liv_261,"Yes, goat and poultry rearing were significant...",825
825,How is livestock production competing with cro...,livestock,livestock,highland,doc_liv_269,The study shows that livestock production is c...,826
826,How can smallholder farms in Ségou and Sikass...,water_management,general,semi_arid,doc_wat_089,Smallholders can adopt small-scale solar irrig...,827
827,How can I start using pelleted feed for my liv...,livestock,livestock,highland,doc_liv_288,Join farmer organizations that collaborate wit...,828


In [129]:
train.head()

,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5


## FETCH ROWS NEEDED FOR TRAINING (COULD BE ALL)

In [183]:
train_600.to_csv('C:\\Users\\user\\Downloads\\last_experiment\\kulal_train_qa.csv')